# UNVERIFIED / INCOMPLETE NOTEBOOK

**未検証・途中までの参考Notebookです。** 全セルの実行成功は確認していません。

**Stopping reason:** Pretrained SFC2Net checkpoint (model_best.pth.tar, 26.2 MB) is access-gated: OneDrive share migrated to SharePoint and redirects to login (shares API 401, Graph API 401, download.aspx error pages); BaiduYun share verification returns errno 2 without login; no public mirror exists (GitHub releases/forks/code search, Hugging Face checked). Minimal reproduction (pretrained inference, MAE 25.51/MSE 38.06) is therefore blocked. An honest data-side partial demonstration was authored and executed clean (zero cell errors) on a fresh CPU Colab runtime: RPC test set from OPIA (611 MB, CRC-verified, 152 images + 152 label mats + test.txt), author RiceDataset/transform pipeline over all images, reference dot-annotation previews, test-count statistics exactly matching the paper (341-1255, mean 648.39; 98555 dots), and the blockwise/log-quantized class-map target scheme with the author's verbatim decode table (round-trip median rel. error 2.52%). Deliverable preserved at deliverables/p-b113edaf7173c

Attempt status: blocked. No manual login validation was performed. This draft may require your own authorized access, fail partway, or contain incomplete code. This attempt does not establish that the paper cannot be reproduced.

Paper: High-Throughput Rice Density Estimation from Transplantation to Tillering Stages Using Deep Networks.

[PhenoPaper record](https://phenopaper.smartbreed-plant-phenotyping-platform.com/papers/p-b113edaf7173c42780d2402708d4ff97)


# SFC²Net on the RPC test set — rice plant counting (partial, data-side demonstration)

**Paper:** Liu, Lu, Li, Cao — *High-Throughput Rice Density Estimation from Transplantation to Tillering Stages Using Deep Networks*, Plant Phenomics 2020, DOI [10.34133/2020/1375957](https://doi.org/10.34133/2020/1375957) (PMCID PMC7706318)
**Code:** [poppinace/sfc2net](https://github.com/poppinace/sfc2net) @ commit `c2ae79984024fcc4e5720cb5f6385bc5cb0fc777` (non-commercial use only)
**Data:** [OPIA record dataId=5](https://ngdc.cncb.ac.cn/opia/dataset/datasets/tables?dataId=5) — the 152-image RPC **test** split with dot annotations (direct zip, 611 MB)

## Scope and execution status — read first / まずお読みください

**English:** This notebook is a **partial demonstration**: it executes the paper's *data side* faithfully
(dataset acquisition, the author's dataset pipeline, reference dot annotations, test-set count statistics,
and the blockwise count-map / log-quantized class-map target scheme with the author's exact decode table).
**Pretrained-model inference is NOT executed**: the author's checkpoint (`model_best.pth.tar`, 26.2 MB) is
only distributed via OneDrive/BaiduYun shares that now require an interactive login (verified 2026-10-07;
redirects go to `login.live.com`; BaiduYun share verification returns `errno 2` without an account), so the
reported counting metrics (MAE 25.51 / MSE 38.06, paper Table 7) **could not be verified here**.

**日本語:** このノートブックは**部分実演**です。データ側（データセット取得、著者パイプライン、参照ドット注釈、
テストセット統計、ブロックごとのカウントマップと対数量子化クラスマップの構成）を実行します。
**学習済みモデルの推論は実行されません**：著者チェックポイントは OneDrive/BaiduYun 共有のみで配布されており、
いずれも対話ログインが必要です（2026-10-07 検証）。よって論文報告値 MAE 25.51 / MSE 38.06 は本ノートブックでは**検証できません**。

- **Runtime:** CPU is sufficient for everything executed here. The blocked inference path needs a GPU (T4) — the author's evaluation is CUDA-only (`hlnet.py:42,68`, `hltest.py:139`).
- **Runtime:** 実行部分は CPU で十分です。ブロックされた推論には GPU(T4) が必要です（著者コードは CUDA 前提）。
- **Run all:** ~5–10 min, ~611 MB download (RPC test zip) + ~26 KB author code.
- **Run all:** 所要 5–10 分、ダウンロード約 611 MB。

## Environment check / 環境確認
The author's `requirements.txt` (opencv-python, numpy, pandas, matplotlib, scipy, pillow) is fully covered
by Colab's preinstalled stack — verified against the pinned import graph of `hltest.py`/`hldataset.py`/`hlnet.py`.
This cell prints the actual versions and fails if CUDA-dependent pieces were needed (they are not in the executed scope).
環境はColab標準スタックで全て揃います。実行範囲にCUDAは不要です。

In [ ]:
import sys, importlib
print('python', sys.version.split()[0])
for m in ['torch', 'cv2', 'scipy', 'matplotlib', 'numpy', 'pandas', 'PIL']:
    mod = importlib.import_module(m)
    print(f'{m:12s}', getattr(mod, '__version__', '?'))
# the author's dataset import path (verified compatible with this stack):
import scipy.ndimage.filters  # noqa: F401  (deprecated namespace, still present)
print('scipy.ndimage.filters import OK')

## Acquire the pinned author code / 著者コードの取得
Partial clone of `poppinace/sfc2net` at the verified commit, inside `/content` (code only; the repo's two PNG
figures are skipped by the sparse checkout). The commit is asserted to match the inspected revision
`c2ae79984024fcc4e5720cb5f6385bc5cb0fc777`.
検証済みコミットにピン留めした部分クローンを `/content` に作成し、コミット一致を assert します。

In [ ]:
import subprocess, sys, os
SHA = 'c2ae79984024fcc4e5720cb5f6385bc5cb0fc777'
REPO = '/content/sfc2net'
if not os.path.exists(REPO):
    subprocess.run(['git', 'init', REPO], check=True)
    def g(*a):
        subprocess.run(['git', '-C', REPO] + list(a), check=True)
    g('remote', 'add', 'origin', 'https://github.com/poppinace/sfc2net')
    g('config', 'extensions.partialClone', 'origin')
    g('fetch', '--depth=1', '--filter=blob:none', 'origin', SHA)
    g('sparse-checkout', 'set', 'hldataset.py', 'hlnet.py', 'hltest.py', 'utils.py',
      'requirements.txt', 'LICENSE.md', 'mixnet')
    g('checkout', '--detach', SHA)
head = subprocess.run(['git', '-C', REPO, 'rev-parse', 'HEAD'],
                      capture_output=True, text=True).stdout.strip()
assert head == SHA, f'unexpected commit {head}'
print('author code pinned at', head)
sys.path.insert(0, REPO)

## Download the RPC test set (611 MB) / RPC テストセットのダウンロード
The paper-linked OPIA record (dataId=5, "RPC", 152 images) exposes a direct public archive at
`download.cncb.ac.cn`. Expected size: 611,052,852 bytes (verified 2026-10-07). The cell streams the
download, checks the byte count against the expected size, and records the elapsed time.
OPIA レコードが公開する直接 URL からダウンロードし、期待サイズと照合します。

In [ ]:
import time, requests, os
URL = 'https://download.cncb.ac.cn/OPIA/RPC_Rice.zip'
EXPECTED_SIZE = 611052852
DST = '/content/RPC_Rice.zip'
t0 = time.time()
with requests.get(URL, stream=True, timeout=180) as r:
    r.raise_for_status()
    print('status', r.status_code, '| content-length', r.headers.get('Content-Length'),
          '| type', r.headers.get('Content-Type'))
    n = 0
    with open(DST, 'wb') as f:
        for chunk in r.iter_content(1 << 20):
            f.write(chunk)
            n += len(chunk)
            if n % (200 << 20) < (1 << 20):
                print(f'  {n/1e6:.0f} MB', flush=True)
elapsed = time.time() - t0
size = os.path.getsize(DST)
print(f'downloaded {size} bytes in {elapsed:.1f}s')
assert size == EXPECTED_SIZE, f'size mismatch: {size} != {EXPECTED_SIZE}'
print('size check OK')

## Validate and extract the archive / アーカイブの検証と展開
Before use: list members, count images/labels, and run a CRC integrity test
(`zipfile.testzip`). Expected: `rice_datasets-test/` with 152 JPEGs, 152 label `.mat`
files, and a 152-line `test.txt` (tab-separated `images/…jpg<TAB>label_mat/…mat`),
matching the layout `hltest.py` expects under `./data/rice_datasets-test/`.
展開前にCRC検証を行い、152枚+152注釈+test.txt の構成を確認します。

In [ ]:
import zipfile
zf = zipfile.ZipFile('/content/RPC_Rice.zip')
names = zf.namelist()
jpgs = [n for n in names if n.lower().endswith('.jpg')]
mats = [n for n in names if n.lower().endswith('.mat')]
txts = [n for n in names if n.lower().endswith('.txt')]
print(f'members={len(names)} jpg={len(jpgs)} mat={len(mats)} txt={len(txts)}')
assert len(jpgs) == 152 and len(mats) == 152, 'unexpected member counts'
assert zf.testzip() is None, 'zip CRC failure'
zf.extractall('/content/rpc')
lines = open('/content/rpc/rice_datasets-test/test.txt').read().splitlines()
print('test.txt lines:', len(lines))
print('first line:', lines[0])
assert len(lines) == 152
DATA_DIR = '/content/rice_datasets-test'
if not os.path.exists(DATA_DIR):
    os.rename('/content/rpc/rice_datasets-test', DATA_DIR)
print('data dir ready:', sorted(os.listdir(DATA_DIR)))

## Input preview with reference annotations / 入力画像と参照注釈のプレビュー
Three deterministic test images (first three lines of `test.txt`) with their **reference dot annotations**
(`annPoints` from `label_mat/*.mat`, one dot per rice plant, paper Sec. 2.2) drawn on the original photos.
These are the author-published ground-truth annotations — not model output.
`test.txt` の先頭3枚について、label_mat の参照ドット注釈（1 plant = 1 dot）を重畳して表示します。

In [ ]:
import numpy as np, scipy.io as sio
from PIL import Image
import matplotlib
import matplotlib.pyplot as plt

lines = open(f'{DATA_DIR}/test.txt').read().splitlines()[:3]
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, line in zip(axes, lines):
    img_rel, mat_rel = line.split('\t')
    img = np.array(Image.open(f'{DATA_DIR}/{img_rel}'))
    ann = sio.loadmat(f'{DATA_DIR}/{mat_rel}')['annPoints']
    ax.imshow(img)
    ax.scatter(ann[:, 0], ann[:, 1], s=2, c='cyan', label='reference dots (annPoints)')
    ax.set_title(f"{os.path.basename(img_rel)} — {ann.shape[0]} plants (reference)", fontsize=9)
    ax.axis('off')
axes[0].legend(loc='lower left', fontsize=8)
plt.tight_layout()
plt.savefig('/content/preview_reference_annotations.png', dpi=150, bbox_inches='tight')
plt.show()
print('saved /content/preview_reference_annotations.png')

## Run the author's dataset pipeline over all 152 test images / 著者パイプラインの実行
`RiceDataset` (author `hldataset.py`) with the exact evaluation transform stack from `hltest.py`
(`Normalize` with the author constants `IMG_MEAN/IMG_STD`, `ToTensor`, `ZeroPadding`).
Each image is resized by ratio 0.25 and its dot target Gaussian-smoothed (sigma=4) exactly as the author
code does; the per-image ground-truth count is the number of `annPoints`. This exercises the author's
faithful preprocessing over the whole test split.
著者の `RiceDataset` と `hltest.py` と同一の変換列で全152枚を処理し、各画像の正解カウントを取得します。

In [ ]:
import time, numpy as np, torch
from torchvision import transforms
from torch.utils.data import DataLoader
from hldataset import RiceDataset, Normalize, ToTensor, ZeroPadding

IMG_SCALE = 1./255
IMG_MEAN = np.array([0.4607, 0.4836, 0.3326]).reshape((1, 1, 3))   # hltest.py constants
IMG_STD = np.array([0.1058, 0.1065, 0.0999]).reshape((1, 1, 3))

transform_test = transforms.Compose([
    Normalize(IMG_SCALE, IMG_MEAN, IMG_STD),
    ToTensor(),
    ZeroPadding(32),          # OUTPUT_STRIDE padding, as in hltest.py
])
testset = RiceDataset(data_dir=DATA_DIR + '/',
                      data_list=DATA_DIR + '/test.txt',
                      ratio=0.25,             # RESIZE_RATIO from hltest.py
                      train=False, transform=transform_test, gauss_kernel=4)
test_loader = DataLoader(testset, batch_size=1, shuffle=False, num_workers=2)
t0 = time.time()
gt_counts = []
for sample in test_loader:
    gt_counts.append(int(sample['gtcount'].item()))
print(f'pipeline over {len(gt_counts)} images in {time.time()-t0:.1f}s')
print('sample image tensor shape:', sample['image'].shape,
      '| normalized range [%.2f, %.2f]' % (sample['image'].min(), sample['image'].max()))
np.save('/content/gt_counts.npy', np.array(gt_counts))
print('saved /content/gt_counts.npy')

## Test-set count statistics vs paper reference / 正解カウント統計の検証
The paper (Sec. 2.2 / Table 1 text) reports RPC test counts **341–1255, mean 648.39** (also recorded in the
prior research flow). We compare the counts recovered through the author pipeline with those reference
values — a quantitative consistency check of both the dataset and the pipeline.
パイプライン経由で得たカウント統計を論文報告値（341–1255, 平均 648.39）と照合します。

In [ ]:
import pandas as pd
gt = np.load('/content/gt_counts.npy')
stats = {
    'images': len(gt),
    'min': int(gt.min()), 'max': int(gt.max()),
    'mean': float(gt.mean()), 'total dots': int(gt.sum()),
}
ref = {'images': 152, 'min': 341, 'max': 1255, 'mean': 648.39, 'total dots': None}
df = pd.DataFrame({'pipeline': stats, 'paper reference': ref})
print(df.to_string())
match = (stats['images'] == ref['images'] and stats['min'] == ref['min']
         and stats['max'] == ref['max'] and abs(stats['mean'] - ref['mean']) < 1.0)
print('consistency with paper reference:', 'PASS' if match else 'CHECK')
assert match, 'test-set statistics deviate from the paper reference'
df.T.to_csv('/content/rpc_test_statistics.csv')
print('saved /content/rpc_test_statistics.csv')

### Count distribution graph / カウント分布グラフ
Distribution of the 152 reference counts (added visualization for this notebook, not an author figure).
152枚の参照カウントの分布（本ノートブック追加の可視化であり、著者の図ではありません）。

In [ ]:
plt.figure(figsize=(7, 4))
plt.hist(gt, bins=30, color='#4c72b0', edgecolor='white')
plt.axvline(gt.mean(), color='r', linestyle='--', label=f'mean {gt.mean():.2f}')
plt.xlabel('plants per image (reference dots)')
plt.ylabel('number of test images')
plt.title('RPC test set — reference count distribution (this notebook)')
plt.legend()
plt.tight_layout()
plt.savefig('/content/rpc_test_count_distribution.png', dpi=150)
plt.show()

## The counting-target scheme: density → blockwise counts → log-quantized class map / カウントターゲット構成
SFC²Net's key idea (paper Sec. 2.3) is to turn dot annotations into **log-space quantized count-interval
class maps** instead of regressing raw counts. The decode direction below is the **author's exact code**
(`class2regression` construction, `hlnet.py:166–173`: `s=0.1`, `q=-2`, 80 classes). The encode direction
(density → blockwise 32×32/stride-8 sums → class index) is an AI implementation of the paper's description —
the training script is not part of the repository.
デコード側は著者コードそのまま、エンコード側は論文記述に基づくAI実装です（学習スクリプトはリポジトリ外）。

In [ ]:
# Author's exact decode table (hlnet.py:166-173), copied verbatim with citation:
step_log, start_log, max_class_number = 0.1, -2, 80   # STEP_DEFAULT, START_DEFAULT, MAX_CLASS_NUMBER
class2regression = np.zeros(max_class_number)
for i in range(1, max_class_number):
    if i == 1:
        lower = 0
    else:
        lower = np.exp((i - 2) * step_log + start_log)
    upper = np.exp((i - 1) * step_log + start_log)
    class2regression[i] = (lower + upper) / 2
print('class2regression[:5] =', class2regression[:5])
print('interval widths grow geometrically: ratio', np.exp(step_log))

### Round-trip check on a real test image / 実画像でのラウンドトリップ検証
For one test image: take the author's Gaussian-smoothed density target, aggregate it into 32×32 blocks at
stride 8 (the same fold/unfold aggregation the author's eval uses in `hlnet.py:68–72`), quantize each block
count to its log-space interval, then decode back through the author's `class2regression` medians.
A small round-trip error validates that the quantization preserves counts.
ある1枚の密度ターゲットからブロック化→量子化→デコードし、復元誤差を確認します。

In [ ]:
import torch.nn.functional as F
line = open(f'{DATA_DIR}/test.txt').read().splitlines()[0]
img_rel, mat_rel = line.split('\t')
raw = np.array(Image.open(f'{DATA_DIR}/{img_rel}'))
h, w = raw.shape[:2]
nh, nw = int(np.ceil(h * 0.25)), int(np.ceil(w * 0.25))
ann = sio.loadmat(f'{DATA_DIR}/{mat_rel}')['annPoints']
from scipy.ndimage import gaussian_filter
density = np.zeros((nh, nw), dtype=np.float32)
for px, py in ann:
    x, y = int(np.floor(px * 0.25)) - 1, int(np.floor(py * 0.25)) - 1
    if x >= w or y >= h:
        continue
    density[y, x] = 1
density = gaussian_filter(density, 4)              # GAUSS_KERNEL=4 (hltest.py)

d = torch.from_numpy(density)[None, None]          # 1,1,H,W
cols = F.unfold(d, kernel_size=32, stride=8)       # author's block aggregation geometry
block_counts = cols.sum(1).view(1, 1, (nh - 32) // 8 + 1, (nw - 32) // 8 + 1)[0, 0].numpy()

edges = np.exp(np.arange(0, max_class_number) * step_log + start_log)  # edges[j] = upper edge of class j+1
classes = np.clip(np.searchsorted(edges, block_counts, side='left') + 1, 1, max_class_number - 1)
decoded = class2regression[classes]

rel = np.abs(decoded - block_counts)[block_counts > 1] / block_counts[block_counts > 1]
print(f'blocks with >1 plant: {len(rel)}')
print(f'median round-trip relative error: {np.median(rel)*100:.2f}%  | mean: {rel.mean()*100:.2f}%')
print(f'density sum {density.sum():.1f} vs annPoints {ann.shape[0]}')

### Visual: density, blockwise counts, decoded class map / 可視化
Side-by-side: the Gaussian density target, the blockwise count map, and the count map recovered from the
quantized class map (author's decode). Brighter = more plants per block.
密度ターゲット、ブロック化カウントマップ、量子化→デコードで復元したカウントマップを並べて表示します。

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
im0 = axes[0].imshow(density, cmap='jet'); axes[0].set_title('Gaussian density target (author pipeline)')
im1 = axes[1].imshow(block_counts, cmap='jet'); axes[1].set_title('blockwise count map (32x32 / stride 8)')
im2 = axes[2].imshow(decoded, cmap='jet')
axes[2].set_title('decoded from quantized class map (author class2regression)')
for ax, im in zip(axes, [im0, im1, im2]):
    ax.axis('off'); plt.colorbar(im, ax=ax, fraction=0.04)
plt.tight_layout()
plt.savefig('/content/classmap_roundtrip.png', dpi=150, bbox_inches='tight')
plt.show()

## Normalization-constant check / 正規化定数の確認
`hltest.py` normalizes with `IMG_MEAN=[0.4607,0.4836,0.3326]`, `IMG_STD=[0.1058,0.1065,0.0999]`
(training-set statistics per the paper). We recompute the statistics of the *test* set at ratio 0.25 —
an approximate consistency check (test ≠ train, so small deviations are expected).
論文の定数（学習セット統計）に対する近似確認としてテストセット統計を再計算します。

In [ ]:
means, stds = [], []
for line in open(f'{DATA_DIR}/test.txt').read().splitlines():
    img_rel, _ = line.split('\t')
    a = np.array(Image.open(f'{DATA_DIR}/{img_rel}').resize((nw, nh))).astype(np.float32) / 255.
    means.append(a.reshape(-1, 3).mean(0)); stds.append(a.reshape(-1, 3).std(0))
test_mean, test_std = np.mean(means, axis=0), np.mean(stds, axis=0)
print('test-set mean (RGB):', np.round(test_mean, 4), '| author IMG_MEAN:', IMG_MEAN.ravel())
print('test-set std  (RGB):', np.round(test_std, 4), '| author IMG_STD :', IMG_STD.ravel())
print('|Δmean| max:', float(np.abs(test_mean - IMG_MEAN.ravel()).max()))

## ⛔ Blocked: pretrained-model inference / ブロック：学習済み推論
**English:** The author checkpoint `model_best.pth.tar` (26.2 MB) is published only through
[BaiduYun (code: nhxk)](https://pan.baidu.com/s/1HIFPbLJYCJy1ux1YO6bVsQ) and
[OneDrive](https://1drv.ms/u/s!AkNf_IPSDakh5zdqa5c8Co5QzB9y?e=EhqoWA). On 2026-10-07 every programmatic
anonymous route was tested and failed: the OneDrive share was migrated to SharePoint and now redirects to a
login page (shares-API 401 `unauthenticated`, Graph API 401), and the BaiduYun share verification returns
`errno 2` without an account. No public mirror exists (GitHub releases/search/forks and Hugging Face checked).
The author's evaluation is CUDA-only, so with the checkpoint a learner would run:

```bash
# author's documented inference (NOT executed in this notebook):
#  1. place model_best.pth.tar at ./snapshots/rice/sfc2net/model_best.pth.tar
#  2. place the test set at ./data/rice_datasets-test/  (as downloaded above)
#  3. python hltest.py   # -> MAE 25.51, MSE 38.06 (paper Table 7)
```

**日本語:** 著者チェックポイントは BaiduYun(コード: nhxk) と OneDrive のみで公開されており、2026-10-07 時点で
いずれも匿名アクセス不可（ログイン要求）であることを確認しました。公開ミラーも存在しません。
チェックポイントを入手できる学習者は、上の著者コマンド（`python hltest.py`）で MAE 25.51 / MSE 38.06 の
検証が可能です。本ノートブックでは**推論を実行しません**。

## Summary, limitations, validation record / まとめと制限
**English:** Executed scope: dataset acquisition + integrity checks, the author's exact preprocessing over
all 152 test images, reference-annotation previews, test-count statistics matching the paper
(341–1255, mean 648.39), and the blockwise/log-quantized class-map target scheme with the author's exact
decode table (round-trip median error printed above). **Not executed:** model inference — the pretrained
checkpoint is access-gated; the reported MAE 25.51 / MSE 38.06 (paper Table 7) could not be verified.
This one-image/one-dataset run is not verification of the paper's full benchmark.

**日本語:** 実行済み: データ取得・整合性確認、著者パイプライン全152枚、参照注釈プレビュー、
カウント統計の論文値との一致確認（341–1255, 平均 648.39）、ブロック化・対数量子化クラスマップ方式の
ラウンドトリップ検証（著者のデコード表をそのまま使用）。**未実行**: 学習済みモデル推論
（チェックポイントがアクセス制限のため取得不可）。MAE 25.51 / MSE 38.06 は検証未了です。

**References:** paper DOI 10.34133/2020/1375957 · code poppinace/sfc2net@`c2ae79984024fcc4e5720cb5f6385bc5cb0fc777`
(non-commercial) · data OPIA dataId=5 (RPC test split). Prior research guidance: PhenoPaper investigation
`p-b113edaf7173c42780d2402708d4ff97-20260930T003755Z-3118082` (unverified AI findings).

In [ ]:
import pandas as pd
summary = pd.DataFrame({
    'item': ['test images', 'reference dot annotations', 'count range / mean (pipeline)',
             'paper reference', 'blockwise/decode round-trip median rel. error',
             'pretrained inference', 'reported MAE / MSE'],
    'value': [152, int(gt.sum()),
              f"{gt.min()}-{gt.max()} / {gt.mean():.2f}",
              '341-1255 / 648.39',
              f"{np.median(rel)*100:.2f}%",
              'BLOCKED (checkpoint access-gated)',
              '25.51 / 38.06 (not verified here)'],
})
print(summary.to_string(index=False))
summary.to_csv('/content/sfc2net_partial_summary.csv', index=False)
print('saved /content/sfc2net_partial_summary.csv')